# **03 - Neon connection and first raw load**

## GOAL
- Test the Python connection to the Neon database.
- First step of the pipeline, if Python can't reach the database, nothing else will work.
- The result is the raw schema and the `loan` table loaded and verified.

Import the libraries used in this notebook.

In [1]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd
import os
import sys
sys.path.append("..")
from src.extract import load_table


.env keeps the credential out of the code and GitHub. GitHub Actions doesn't load .env automatically like VS code does, so we call load_dotenv explicitly.

In [2]:
load_dotenv("../.env")

True

os.getenv reads the `DATABASE_URL` stored in .env (the Neon access key). Never print db_url because it would show the password, so we only check it with `is not None`. 

In [3]:
db_url = os.getenv("DATABASE_URL")

In [4]:
db_url is not None 

True

The engine variable holds the connection setup. We use the key and pool_pre_ping, because Neon goes to sleep after about 5 idle minutes and the ping avoids using a stale connection. 

SQLAlchemy 2.1 uses psycopg 3 by default, so we switched from psycopg 2 to psycopg 3.

In [5]:
engine = create_engine(db_url, pool_pre_ping=True) 

Minimal connection test. If it returns 1, Neon answered.  

In [6]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1")).scalar()
    print(result)

1


Here we use `engine.begin()` because it saves the change automatically. With `connect()` we could do it too, but we would have to write `conn.commit()` at the end for the change to take place.
We create a schema called raw to store all the raw data. Later we will create processed to store the clean data.
We use `IF NOT EXISTS` so we can re-run the cell without errors.

In [7]:
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS raw"))

Here we load the loan table into Neon with `to_sql` using the following parameters:
* **loan:** The table's name
* **engine:** The Neon access setup 
* **schema:** the raw schema
* **if_exists:** If the table already exists, we can re-run without duplicating it.
* **index:** avoids loading a column with index numeration.

psycopg 3 doesn't report how many rows were loaded (it returns -1), so we verify it in the next cell.

In [8]:
load_table("loan").to_sql("loan", engine, schema="raw", if_exists="replace", index=False)

-1

With COUNT we verify the 682 rows in the loan table. No error is not enough, so we check that all the rows arrived.

In [9]:
pd.read_sql("SELECT COUNT(*) FROM raw.loan", engine)

,count
0,682
